# Phần 1: Lý thuyết và Minh họa Data Fitting

## Thành viên thực hiện
- **Nguyên**: Cài đặt thuật toán OLS
- **Nam**: Mô phỏng Monte Carlo & K-fold CV

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression

# Import module
import ols_implementation as ols

# Fix random seed để kết quả tái lập được
np.random.seed(42)

### 1. Khởi tạo dữ liệu giả lập (Synthetic Data)
Để minh họa và kiểm chứng các hàm đã cài đặt, chúng ta sẽ tạo một bộ dữ liệu giả lập đơn giản với $n=100$ quan trắc và $p=2$ biến đặc trưng.

Mô hình thực tế:
$$y = 2.0 + 1.5 x_1 - 2.0 x_2 + \epsilon$$
với $\epsilon \sim \mathcal{N}(0, 0.5^2)$

In [3]:
n = 100
X_synth = np.random.randn(n, 2)
beta_true = np.array([2.0, 1.5, -2.0]) # Intercept = 2.0

# Thêm cột 1 vào X để nhân ma trận
X_design_synth = np.column_stack((np.ones(n), X_synth))
y_synth = X_design_synth @ beta_true + np.random.randn(n) * 0.5

### 2. Nghiệm Ordinary Least Squares (OLS)

**(a) Trình bày bài toán và Hàm mất mát:**
Mô hình hồi quy tuyến tính có dạng $y = X\beta + \epsilon$, trong đó $\epsilon$ là nhiễu ngẫu nhiên.
Phương pháp OLS tìm kiếm vector tham số $\hat{\beta}$ sao cho Tổng bình phương phần dư (Residual Sum of Squares - RSS) đạt giá trị nhỏ nhất.

Hàm mất mát RSS được định nghĩa là:
$$RSS(\beta) = ||y - X\beta||_2^2 = (y - X\beta)^T(y - X\beta)$$

**(b) Chứng minh nghiệm OLS (Normal Equations):**
Ta tiến hành khai triển biểu thức $RSS(\beta)$:
$$RSS(\beta) = y^Ty - y^TX\beta - (X\beta)^Ty + (X\beta)^T(X\beta)$$
$$RSS(\beta) = y^Ty - y^TX\beta - \beta^TX^Ty + \beta^TX^TX\beta$$

Vì $y^TX\beta$ là một số vô hướng (scalar), nên chuyển vị của nó bằng chính nó: $(y^TX\beta)^T = \beta^TX^Ty$. Do đó ta có thể gộp lại:
$$RSS(\beta) = y^Ty - 2\beta^TX^Ty + \beta^TX^TX\beta$$

Để tìm giá trị $\beta$ làm cực tiểu hóa RSS, ta lấy đạo hàm riêng (gradient) của $RSS(\beta)$ theo $\beta$ và cho bằng vector 0:
$$\nabla_\beta RSS(\beta) = -2X^Ty + 2X^TX\beta = 0$$

Chuyển vế, ta được hệ phương trình chuẩn (Normal Equations):
$$X^TX\beta = X^Ty$$

Giả sử ma trận thiết kế $X$ có các cột độc lập tuyến tính (full column rank), khi đó ma trận vuông $X^TX$ là khả nghịch. Nhân cả hai vế với $(X^TX)^{-1}$, ta thu được nghiệm OLS duy nhất:
$$\hat{\beta}_{OLS} = (X^TX)^{-1}X^Ty$$

**(c) Cài đặt Python & (d) Minh họa:**
Sử dụng hàm `ols_fit` đã được cài đặt từ đầu bằng ma trận thư viện NumPy (tương đương công thức toán học đã chứng minh ở trên).

In [5]:
# Gọi hàm ols_fit từ module ols_implementation
res = ols.ols_fit(X_synth, y_synth, add_intercept=True)

print("=== KẾT QUẢ TỰ CÀI ĐẶT (ols_fit) ===")
print(f"Hệ số Beta (Intercept, x1, x2): {res['beta']}")
print(f"R^2: {res['R2']:.4f}")

=== KẾT QUẢ TỰ CÀI ĐẶT (ols_fit) ===
Hệ số Beta (Intercept, x1, x2): [ 2.04639669  1.59536017 -2.08607105]
R^2: 0.9555


**(d) Kiểm chứng với Scikit-learn:**

In [6]:
# Dùng sklearn để đối chiếu
sk_model = LinearRegression()
sk_model.fit(X_synth, y_synth)

print("=== KẾT QUẢ TỪ SKLEARN ===")
print(f"Hệ số Beta (Intercept, x1, x2): [{sk_model.intercept_}, {sk_model.coef_[0]}, {sk_model.coef_[1]}]")
print(f"R^2: {sk_model.score(X_synth, y_synth):.4f}")

# Nhận xét: Kết quả của hàm tự cài đặt hoàn toàn khớp với thư viện scikit-learn.

=== KẾT QUẢ TỪ SKLEARN ===
Hệ số Beta (Intercept, x1, x2): [2.046396690621326, 1.595360168680333, -2.086071051549262]
R^2: 0.9555


### 3. Ma Trận Chiếu (Hat Matrix)
**(a) Công thức toán học:**
Ma trận Hat $H$ chiếu vector $y$ lên không gian cột của $X$:
$$H = X(X^T X)^{-1}X^T$$
Tính chất đặc trưng của ma trận chiếu là idempotent: $H^2 = H$.

**(b) Cài đặt Python & (c) Minh họa & (d) Kiểm chứng:**

In [7]:
H = ols.hat_matrix(X_design_synth)

print(f"Kích thước ma trận H: {H.shape}")
print(f"Kiểm tra tính idempotent (H^2 == H): {ols.check_idempotent(H)}")

Kích thước ma trận H: (100, 100)
Kiểm tra tính idempotent (H^2 == H): True


### 4. Đánh giá mô hình và Suy diễn thống kê (Inference)
**(a) Công thức toán học:**

Phương sai nhiễu được ước lượng: $\hat{\sigma}^2 = \frac{RSS}{n-p-1}$
Ma trận hiệp phương sai của $\hat{\beta}$: 

$$Var(\hat{\beta}_{OLS}) = \sigma^2 (X^T X)^{-1}$$

**(b) & (c) Minh họa bằng hàm `coef_inference` và `model_metrics`:**

In [8]:
# Đánh giá tổng thể mô hình
metrics = ols.model_metrics(y_synth, res['y_hat'], p=3)
print("=== CÁC CHỈ SỐ ĐÁNH GIÁ TỔNG THỂ ===")
for k, v in metrics.items():
    print(f"{k}: {v}")

print("\n=== SUY DIỄN THỐNG KÊ CHO CÁC HỆ SỐ ===")
# Bảng suy diễn thống kê cho từng hệ số
df_inference = ols.coef_inference(X_design_synth, y_synth, res['beta'], res['sigma2'])
display(df_inference)

=== CÁC CHỈ SỐ ĐÁNH GIÁ TỔNG THỂ ===
RSS: 27.750867015891654
TSS: 624.1322425323915
R2: 0.9555368796470222
R2_adj: 0.9546201142789196
F_stat: 1042.2916406895145
p_value_F: 2.6791496391633584e-66
n: 100
p: 3

=== SUY DIỄN THỐNG KÊ CHO CÁC HỆ SỐ ===


,beta,se,t,p,ci_lower,ci_upper
0,2.046397,0.054017,37.884433,6.188048e-60,1.939188,2.153605
1,1.595360,0.062810,25.399759,1.193545e-44,1.470700,1.720021
2,-2.086071,0.053847,-38.741042,8.080565e-61,-2.192942,-1.979201


### 5. Đa cộng tuyến (Variance Inflation Factor - VIF)
**(a) Công thức toán học:**
Hệ số phóng đại phương sai cho biến thứ $j$:
$$VIF_j = \frac{1}{1 - R_j^2}$$
Trong đó $R_j^2$ là hệ số xác định khi hồi quy $x_j$ theo các biến độc lập còn lại. Nếu $VIF > 10$, mô hình có hiện tượng đa cộng tuyến nghiêm trọng.

**(b), (c) & (d) Minh họa & Kiểm chứng:**
Ta sẽ tạo ra một biến $x_3$ có tương quan tuyến tính rất mạnh với $x_1$ để kiểm tra hàm.

In [9]:
# Tạo x3 gần như bằng x1 * 2
x3 = X_synth[:, 0] * 2 + np.random.randn(n) * 0.05
X_multicollinear = np.column_stack((X_synth, x3))

vif_vals, r2_dict = ols.vif(X_multicollinear, add_intercept=True)

print("VIF của x1:", vif_vals[1])
print("VIF của x2:", vif_vals[2])
print("VIF của x3:", vif_vals[3])

# Nhận xét: VIF của x1 và x3 rất cao (>10) phản ánh chính xác sự đa cộng tuyến do ta cố tình tạo ra.

VIF của x1: 1517.8759740405924
VIF của x2: 1.0018223522970675
VIF của x3: 1517.8055451888488
